<a href="https://colab.research.google.com/github/Karsuman4298/Generative-AI/blob/main/nanoVLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch,math,random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from PIL import Image,ImageDraw
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

In [6]:
IMG_SIZE=32
EMBED_DIM=64
ATTENTION_HEAD=4
BATCH_SIZE=12
EPOCH=10
LR=3e-4
TEMPERATURE=0.08
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')




# SYNTHETIC DATASET

In [7]:
color=['red','green','blue','yellow','purple','orange','pink','brown','grey']
shapes=['square','circle','rectangle']
positions=['left','center','right','top','bottom','top-left','top-right','bottom-left','bottom-right']

# DRAWING IMAGE SHAPES

In [8]:
def draw_sample(color, position, shape, img_size=IMG_SIZE):
  img=Image.new('RGB',(img_size,img_size),'white')
  draw=ImageDraw.Draw(img)
  margin=6
  h=w=img_size-2*margin

  # Calculate x coordinates
  if 'left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'top-left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'bottom-left' in position:
      x0 = margin
      x1 = margin + w // 2
  elif 'right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  elif 'top-right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  elif 'bottom-right' in position:
      x0 = margin + w // 2
      x1 = img_size - margin
  else: # center or vertical positions
      x0 = margin + w // 4
      x1 = margin + 3 * w // 4

  # Calculate y coordinates
  if 'top' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'top-left' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'top-right' in position:
      y0 = margin
      y1 = margin + h // 2
  elif 'bottom' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  elif 'bottom-left' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  elif 'bottom-right' in position:
      y0 = margin + h // 2
      y1 = img_size - margin
  else: # center or horizontal positions
      y0 = margin + h // 4
      y1 = margin + 3 * h // 4

  if shape == 'square':
      draw.rectangle([x0, y0, x1, y1], fill=color, outline='black')
  elif shape == 'circle':
      draw.ellipse([x0, y0, x1, y1], fill=color, outline='black')
  else: # triangle
      draw.polygon([(x0 + (x1 - x0) // 2, y0), (x0, y1), (x1, y1)], fill=color, outline='black')
  return img

In [12]:
class ShapeDataset:
    def __init__(self, image_size=64):
        self.image = []
        self.captions = []
        self.image_size = image_size

        for c in color:          # Fixed: changed 'colors' to 'color'
            for p in positions:
                for s in shapes:
                    img_np = draw_sample(c, p, s, img_size=image_size)
                    # Convert to tensor (CHW, normalized to [0,1])
                    img_tensor = (
                        torch.from_numpy(np.asarray(img_np))
                        .permute(2, 0, 1)      # HWC → CHW
                        .float() / 255.0       # ← fixed: / instead of //
                    )
                    self.image.append(img_tensor)
                    self.captions.append(f"{c} {s} {p}")

        # Build vocabulary
        self.vocab, self.word2idx = self.build_vocab(self.captions)

    def build_vocab(self, texts):
        words = sorted({word for text in texts for word in text.split()})
        vocab = ['[CLS]'] + words
        w2i = {w: i for i, w in enumerate(vocab)}
        return vocab, w2i

    def encode_text(self, text):
        toks = [self.word2idx['[CLS]']] + [self.word2idx[word] for word in text.split()]
        return torch.tensor(toks, dtype=torch.long)

    def __len__(self):
        return len(self.image)

    def __getitem__(self, idx):
        return self.image[idx], self.encode_text(self.captions[idx])

In [16]:
full_ds=ShapeDataset()
VOCAB_SIZE=len(full_ds.vocab)
print(VOCAB_SIZE)
print(full_ds.vocab)
print(full_ds.captions)


22
['[CLS]', 'blue', 'bottom', 'bottom-left', 'bottom-right', 'brown', 'center', 'circle', 'green', 'grey', 'left', 'orange', 'pink', 'purple', 'rectangle', 'red', 'right', 'square', 'top', 'top-left', 'top-right', 'yellow']
['red square left', 'red circle left', 'red rectangle left', 'red square center', 'red circle center', 'red rectangle center', 'red square right', 'red circle right', 'red rectangle right', 'red square top', 'red circle top', 'red rectangle top', 'red square bottom', 'red circle bottom', 'red rectangle bottom', 'red square top-left', 'red circle top-left', 'red rectangle top-left', 'red square top-right', 'red circle top-right', 'red rectangle top-right', 'red square bottom-left', 'red circle bottom-left', 'red rectangle bottom-left', 'red square bottom-right', 'red circle bottom-right', 'red rectangle bottom-right', 'green square left', 'green circle left', 'green rectangle left', 'green square center', 'green circle center', 'green rectangle center', 'green squar

# Train - val

In [20]:
train_size=int(0.8 * len(full_ds.captions))
val_size=len(full_ds) - train_size
train_ds,val_ds=torch.utils.data.random_split(full_ds,[train_size,val_size])

In [21]:
train_size,val_size

(194, 49)